# LangSmith, LangGraph, and the Modern Agent Stack

This notebook shows a current, production-minded pattern for building agentic AI systems with observability, graph orchestration, structured outputs, and safer tool use.


## What changed in the modern stack

Useful current pieces to know:

- LangSmith for tracing, datasets, and evaluation
- LangGraph for multi-step workflows and controlled branching
- Typed outputs with Pydantic or dataclasses
- Tool calling with narrow, testable functions
- Streaming responses for better UX
- Human-in-the-loop review for risky actions
- Local model swaps for demos, including GPU or vLLM later


In [ ]:
# LangSmith tracing usually starts with environment variables.
# Keep secrets out of notebooks; load them from your environment or .env file.

import os

os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "tips-tricks-agentic-ai")

{k: os.environ[k] for k in ['LANGCHAIN_TRACING_V2', 'LANGSMITH_TRACING', 'LANGCHAIN_PROJECT']}


## LangSmith in practice

Treat LangSmith as the observation layer for prompts, tool calls, chain steps, and failures.

The most useful habit is to inspect traces after both success and failure cases.


In [ ]:
from dataclasses import dataclass

@dataclass
class TicketSummary:
    title: str
    priority: str
    owner: str
    reason: str

def summarize_ticket(text: str) -> TicketSummary:
    if 'billing' in text.lower():
        return TicketSummary('Billing slowdown', 'high', 'finance-ops', 'Billing workflow symptoms detected')
    if 'login' in text.lower() or 'auth' in text.lower():
        return TicketSummary('Authentication delay', 'high', 'identity', 'Auth symptoms detected')
    return TicketSummary('General triage', 'medium', 'support', 'Needs deeper inspection')

summarize_ticket('Users see login timeouts after shift change.')


## LangGraph workflow skeleton

A graph gives you explicit stages and makes it easier to add retries, checkpoints, and human approval.


In [ ]:
# This is the modern pattern: keep the workflow explicit.
# If LangGraph is installed, this shape maps cleanly to a StateGraph.

workflow = [
    'ingest',
    'retrieve',
    'reason',
    'compose',
    'review',
]

workflow


In [ ]:
try:
    from langgraph.graph import StateGraph, START, END
except Exception as exc:
    StateGraph = None
    START = END = None
    print('LangGraph not installed in this environment, but the workflow pattern still stands.')

# In a real app, you would wire state transitions here.
StateGraph


## Updated tricks that matter

- Use structured outputs so downstream code can validate the response
- Put retrieval before generation
- Keep tool inputs and outputs small
- Cache repeated retrieval where it is safe
- Add a review step for high-risk actions
- Measure latency, token usage, and escalation rate
- Keep the local fallback model path separate from the production model path


In [ ]:
def build_agent_reply(summary: TicketSummary, citations: list[str]) -> dict:
    return {
        'summary': summary.title,
        'priority': summary.priority,
        'owner': summary.owner,
        'citations': citations,
        'action': 'escalate if priority is high; otherwise keep in review',
    }

build_agent_reply(summarize_ticket('billing retries and login timeouts'), ['doc-12', 'doc-18'])


## Production-minded advice

If you want the notebook path to become an app later, add these layers early:

- trace collection
- eval datasets
- typed schemas
- human approvals
- a stable tool interface
- a local demo backend plus a scalable hosted backend


## Notes on modern model usage

For demos, you can keep using small local models. If you have a GPU, the code comments show where a vLLM endpoint can replace the model backend later.
